In [157]:
import numpy as np

In [158]:
import pandas as pd

course = pd.read_csv("Courses.csv")

### index 칼럼 drop

In [159]:
course = course.drop(columns=["index"])

### Incomplete_flag 제거 관련 코드

In [160]:
# 전체 (incomplete 포함)
overall_rate = course["certified"].mean()

# incomplete 제거 → incomplete_flag가 1이 아닌 값, 즉 NaN인 행만 남기기
clean_rate = course[course["incomplete_flag"].isna()]["certified"].mean()

print("Overall completion rate:", overall_rate)
print("After removing incomplete:", clean_rate)

Overall completion rate: 0.027586884570872418
After removing incomplete: 0.0326723686958965


In [161]:
clean_data = course[course["incomplete_flag"].isna()]

print(clean_data.shape)
print(clean_data["certified"].mean())

(540977, 20)
0.0326723686958965


In [162]:
incomplete_data = course[course["incomplete_flag"] == 1]

print(incomplete_data.shape)
print(incomplete_data["certified"].mean())

(100161, 20)
0.00011980711055201126


In [163]:
overall_rate = course["certified"].mean()
complete_rate = course[course["incomplete_flag"].isna()]["certified"].mean()
incomplete_rate = course[course["incomplete_flag"] == 1]["certified"].mean()

print("Overall completion rate:", overall_rate)
print("Complete-data completion rate:", complete_rate)
print("Incomplete-data completion rate:", incomplete_rate)

Overall completion rate: 0.027586884570872418
Complete-data completion rate: 0.0326723686958965
Incomplete-data completion rate: 0.00011980711055201126


In [164]:
course_clean = course[course["incomplete_flag"].isna()].copy()
course_clean["start_time_DI"] = pd.to_datetime(course_clean["start_time_DI"], errors="coerce")
course_clean["last_event_DI"] = pd.to_datetime(course_clean["last_event_DI"], errors="coerce")

In [165]:
print("Before:", course.shape)
print("After:", course_clean.shape)

Before: (641138, 20)
After: (540977, 20)


### "start_time_DI" > last_event_DI" 경우 처리

In [166]:
# 제거 개수 확인
removed_count = (course_clean["start_time_DI"] > course_clean["last_event_DI"]).sum()

print("Removed rows:", removed_count)

Removed rows: 1341


In [167]:
# 제거 대상 확인
invalid_rows = course_clean[
    course_clean["start_time_DI"] > course_clean["last_event_DI"]
]

invalid_rows.head()

,course_id,userid_DI,registered,viewed,explored,certified,final_cc_cname_DI,LoE_DI,YoB,gender,grade,start_time_DI,last_event_DI,nevents,ndays_act,nplay_video,nchapters,nforum_posts,roles,incomplete_flag
244,HarvardX/CS50x/2012,MHxPC130464231,1,1,0,0,India,NaN,NaN,NaN,0,2013-08-18,2013-03-16,1.0,1.0,NaN,1.0,0,NaN,NaN
261,HarvardX/CS50x/2012,MHxPC130216109,1,1,0,0,Canada,NaN,NaN,NaN,0.0,2013-08-17,2013-03-15,1.0,1.0,NaN,1.0,0,NaN,NaN
775,HarvardX/CS50x/2012,MHxPC130591769,1,1,1,0,Ukraine,NaN,NaN,NaN,0,2013-08-27,2013-05-24,51.0,3.0,NaN,8.0,0,NaN,NaN
930,HarvardX/ER22x/2013_Spring,MHxPC130019081,1,1,0,0,United Kingdom,NaN,NaN,NaN,NaN,2013-07-30,2013-04-27,862.0,16.0,NaN,14.0,2,NaN,NaN
1091,HarvardX/CS50x/2012,MHxPC130295771,1,1,0,0,Other Europe,NaN,NaN,NaN,0,2013-07-19,2013-04-07,1.0,1.0,NaN,2.0,0,NaN,NaN


In [168]:
course_clean = course_clean[
    (course_clean["last_event_DI"].isna()) |
    (course_clean["start_time_DI"] <= course_clean["last_event_DI"])
]

### "start_time_DI" == last_event_DI" 경우 처리

In [169]:
print("Invalid rows count:", (course_clean["start_time_DI"] == course_clean["last_event_DI"]).sum())

Invalid rows count: 142740


In [170]:
same_time = course_clean[
    course_clean["start_time_DI"] == course_clean["last_event_DI"]
]

In [171]:
problem_cases = same_time[
    (same_time["explored"] == 1) | (same_time["certified"] == 1)
]

print(problem_cases.shape)
problem_cases.head()

(418, 20)


,course_id,userid_DI,registered,viewed,explored,certified,final_cc_cname_DI,LoE_DI,YoB,gender,grade,start_time_DI,last_event_DI,nevents,ndays_act,nplay_video,nchapters,nforum_posts,roles,incomplete_flag
568,HarvardX/CS50x/2012,MHxPC130046325,1,1,1,0,United Kingdom,NaN,NaN,NaN,0,2013-03-11,2013-03-11,18.0,1.0,NaN,6.0,0,NaN,NaN
702,HarvardX/CS50x/2012,MHxPC130523644,1,1,1,0,India,NaN,NaN,NaN,0,2013-03-18,2013-03-18,14.0,1.0,NaN,7.0,0,NaN,NaN
1229,HarvardX/ER22x/2013_Spring,MHxPC130392295,1,1,1,0,India,NaN,NaN,NaN,NaN,2013-06-17,2013-06-17,421.0,9.0,NaN,19.0,0,NaN,NaN
1295,HarvardX/CS50x/2012,MHxPC130395466,1,1,1,0,United States,NaN,NaN,NaN,0,2013-06-02,2013-06-02,51.0,3.0,NaN,7.0,0,NaN,NaN
1976,HarvardX/CS50x/2012,MHxPC130226479,1,1,1,0,India,NaN,NaN,NaN,0.0,2013-07-20,2013-07-20,1.0,1.0,NaN,11.0,0,NaN,NaN


In [172]:
print("Explored count:", (same_time["explored"] == 1).sum())
print("Certified count:", (same_time["certified"] == 1).sum())

Explored count: 372
Certified count: 54


In [173]:
pd.crosstab(
    same_time["explored"],
    same_time["certified"]
)

certified,0,1
explored,,
0,142322,46
1,364,8


In [174]:
course_valid = course_clean[
    ~(
        (course_clean["start_time_DI"] == course_clean["last_event_DI"]) &
        (
            (course_clean["explored"] == 1) |
            (course_clean["certified"] == 1)
        )
    )
]

In [175]:
print("Before:", course_clean.shape)
print("After:", course_valid.shape)
removed_count = course_clean.shape[0] - course_valid.shape[0]
print("Removed rows:", removed_count)

Before: (539636, 20)
After: (539218, 20)
Removed rows: 418


### last_event_DI가 결측일 때 조건에 따라 값을 다르게 보정하는 작업
- last_event_DI가 결측인 행들 중
- 일반 결측 → 활동성 지표 0 처리, last_event_DI는 그대로 null
- 그중 viewed == 1 인 1,807명 →
- nevents = 1, ndays_act = 1, 나머지 활동성 지표는 0, last_event_DI = start_time_DI

In [176]:
# 복사본
course_fixed = course_valid.copy()

# 조건 정의
mask_last_missing = course_fixed["last_event_DI"].isna()
mask_viewed_missing = mask_last_missing & (course_fixed["viewed"] == 1)
mask_non_viewed_missing = mask_last_missing & (course_fixed["viewed"] != 1)

# 활동성 관련 컬럼 지정
activity_cols = ["nevents", "ndays_act", "nplay_video", "nchapters", "nforum_posts"]

# viewed==1 인 경우:
# nevents=1, ndays_act=1, 나머지는 0, last_event_DI는 start_time_DI와 같은 날로 처리
course_fixed.loc[mask_viewed_missing, "nevents"] = 1
course_fixed.loc[mask_viewed_missing, "ndays_act"] = 1

other_activity_cols = [col for col in activity_cols if col not in ["nevents", "ndays_act"]]
course_fixed.loc[mask_viewed_missing, other_activity_cols] = 0

course_fixed.loc[mask_viewed_missing, "last_event_DI"] = course_fixed.loc[mask_viewed_missing, "start_time_DI"]

# viewed!=1 인 일반 결측:
# 활동성 지표는 모두 0, last_event_DI는 null 유지
course_fixed.loc[mask_non_viewed_missing, activity_cols] = 0
course_fixed.loc[mask_non_viewed_missing, "last_event_DI"] = pd.NaT

# 확인
print("Total missing last_event_DI:", mask_last_missing.sum())
print("Viewed=1 among missing:", mask_viewed_missing.sum())
print("Viewed!=1 among missing:", mask_non_viewed_missing.sum())
print(course_fixed.loc[mask_last_missing, ["viewed", "start_time_DI", "last_event_DI"] + activity_cols].head())

Total missing last_event_DI: 93353
Viewed=1 among missing: 1807
Viewed!=1 among missing: 91546
    viewed start_time_DI last_event_DI  nevents  ndays_act  nplay_video  \
25       0    2012-09-04           NaT      0.0        0.0          0.0   
32       0    2012-12-23           NaT      0.0        0.0          0.0   
35       0    2012-09-05           NaT      0.0        0.0          0.0   
40       0    2012-08-27           NaT      0.0        0.0          0.0   
45       0    2012-09-17           NaT      0.0        0.0          0.0   

    nchapters  nforum_posts  
25        0.0             0  
32        0.0             0  
35        0.0             0  
40        0.0             0  
45        0.0             0  


In [177]:
course_fixed["last_event_DI"].isna().sum()

np.int64(91546)

In [178]:
print("Original missing:", course_valid["last_event_DI"].isna().sum())
print("After processing:", course_fixed["last_event_DI"].isna().sum())
print("Filled count:", 
      course_valid["last_event_DI"].isna().sum() - course_fixed["last_event_DI"].isna().sum())

Original missing: 93353
After processing: 91546
Filled count: 1807


### 정상 흐름:
- viewed → explored → certified
- 이 순서를 깨는 데이터 제거

In [179]:
course_final = course_fixed[
    ~(
        ((course_fixed["explored"] == 1) & (course_fixed["viewed"] != 1)) |
        ((course_fixed["certified"] == 1) & (course_fixed["explored"] != 1))
    )
]

In [180]:
removed_count = course_fixed.shape[0] - course_final.shape[0]
print("Removed rows:", removed_count)

Removed rows: 646


In [181]:
# 제거 대상만 따로 보기
invalid_funnel = course_fixed[
    ((course_fixed["explored"] == 1) & (course_fixed["viewed"] != 1)) |
    ((course_fixed["certified"] == 1) & (course_fixed["explored"] != 1))
]

invalid_funnel.head()

,course_id,userid_DI,registered,viewed,explored,certified,final_cc_cname_DI,LoE_DI,YoB,gender,grade,start_time_DI,last_event_DI,nevents,ndays_act,nplay_video,nchapters,nforum_posts,roles,incomplete_flag
361,HarvardX/ER22x/2013_Spring,MHxPC130306723,1,1,0,1,United Kingdom,NaN,NaN,NaN,0.62,2012-12-19,2013-08-15,494.0,24.0,NaN,10.0,0,NaN,NaN
1416,HarvardX/PH278x/2013_Spring,MHxPC130550925,1,1,0,1,United States,NaN,NaN,NaN,0.53,2013-01-20,2013-06-24,906.0,13.0,69.0,4.0,0,NaN,NaN
2378,HarvardX/ER22x/2013_Spring,MHxPC130511497,1,1,0,1,Other Middle East/Central Asia,NaN,NaN,NaN,0.92,2013-06-07,2013-06-29,290.0,3.0,NaN,9.0,0,NaN,NaN
5588,HarvardX/ER22x/2013_Spring,MHxPC130083734,1,1,0,1,Brazil,NaN,NaN,NaN,0.63,2013-01-23,2013-08-16,663.0,14.0,NaN,9.0,0,NaN,NaN
6643,HarvardX/ER22x/2013_Spring,MHxPC130536126,1,1,0,1,Colombia,NaN,NaN,NaN,1,2013-01-31,2013-09-02,680.0,27.0,NaN,12.0,0,NaN,NaN


### 2013-11-17일 로그 → 5,637개
→ 특정 강의에 대해서 registered만 존재하고, 나머지 활동성 지표들이 모두 null 처리 되어있음.
→ 여러가지 가능성이 있지만, 데이터 수집 시점으로 cutoff 처리하면서 나머지 활동성 지표들을 null 처리했다고 가정하고 제거

In [182]:
course_final.shape

(538572, 20)

In [183]:
course_final = course_final[
    course_final["last_event_DI"] != pd.Timestamp("2013-11-17")
]

### nchapter null값
0으로 처리

In [184]:
course_final["nchapters"] = course_final["nchapters"].fillna(0)

In [185]:
print(course_final["nchapters"].isna().sum())

0


### nplay_video 컬럼 삭제

In [186]:
course_final = course_final.drop(columns=["nplay_video"])

### roles, incomplete_flag 삭제

In [187]:
course_final = course_final.drop(columns=["roles"])
course_final = course_final.drop(columns=["incomplete_flag"])

### YoB_DI와 start_time_DI를 이용해서 age 생성
년도로 계산하는 것이라, 정확한 만 나이가 아니라, 단순 분석용 연령값임.

In [188]:
# datetime 변환
course_final["start_time_DI"] = pd.to_datetime(course_final["start_time_DI"], errors="coerce")

# 시작 연도 추출
course_final["start_year"] = course_final["start_time_DI"].dt.year

# age 생성
# YoB_DI가 숫자형이 아닐 수도 있으니 먼저 숫자로 변환
course_final["YoB"] = pd.to_numeric(course_final["YoB"], errors="coerce")
course_final["age"] = course_final["start_year"] - course_final["YoB"]

In [189]:
print("Min age:", course_final["age"].min())
print("Max age:", course_final["age"].max())

Min age: 0.0
Max age: 82.0


### age를 구간화한 새 칼럼 생성
20대 밑, 20대, 30대, 40대 50대, 60대 이상

In [190]:
# age 구간화
age_bins = [0, 19, 29, 39, 49, 59, 100]
age_labels = ["under_20", "20s", "30s", "40s", "50s", "60_plus"]

course_final["age_group"] = pd.cut(
    course_final["age"],
    bins=age_bins,
    labels=age_labels
)

### grade를 이용해 exam_flag 생성

In [191]:
# grade가 있으면 시험 친 사람(1), null이면 안 친 사람(0)
course_final["exam_flag"] = np.where(course_final["grade"].isna(), 0, 1)

### LoE_DI, YoB, gender → null을 "unknown"으로 처리

In [192]:
print(course_final[["LoE_DI", "YoB", "gender","age"]].isna().sum())

LoE_DI    87702
YoB       80898
gender    72726
age       80898
dtype: int64


In [193]:
cols = ["LoE_DI", "YoB", "gender", "age"]
course_final[cols] = course_final[cols].fillna("unknown")

In [194]:
print(course_final[["LoE_DI", "YoB", "gender","age"]].isna().sum())

LoE_DI    0
YoB       0
gender    0
age       0
dtype: int64


In [195]:
course_final.to_csv('course_final.csv', index=False)